In [ ]:
test

In [ ]:
,CONCAT_WS(' - ', sc.ctv3_code, br.form_ans_bridge_form_ans_conformed_answer) AS form_ans_src_answer

In [ ]:
LEFT JOIN silver_rdm_form_answer_bridging br
    ON TRIM(LOWER(sc.ctv3_code)) = TRIM(LOWER(br.form_ans_bridge_src_id))
   AND TRIM(LOWER(CONCAT('SONE', sc.id_organisation_source))) = TRIM(LOWER(br.form_ans_bridge_src_sys_inst_src_id))
   AND TRIM(LOWER(COALESCE(selc.question_heading, dermc.question_heading))) = TRIM(LOWER(br.form_ans_bridge_form_ques_src_name))

In [ ]:
%%sql

SELECT
    form_ans_bridge_src_id,
    form_ans_bridge_src_sys_inst_src_id,
    form_ans_bridge_form_ques_src_name,
    COUNT(*) AS bridge_rows,
    COUNT(DISTINCT form_ans_bridge_form_ans_conformed_answer) AS distinct_conformed_answers,
    COUNT(DISTINCT form_ans_bridge_form_ques_src_form_type) AS distinct_form_types,
    COUNT(DISTINCT form_ans_bridge_form_ques_src_status) AS distinct_statuses
FROM silver_rdm_form_answer_bridging
WHERE form_ans_bridge_src_sys_inst_src_id LIKE 'SONE%'
GROUP BY
    form_ans_bridge_src_id,
    form_ans_bridge_src_sys_inst_src_id,
    form_ans_bridge_form_ques_src_name
HAVING COUNT(*) > 1
ORDER BY bridge_rows DESC;

In [ ]:
%%sql

WITH br_dedup AS (
    SELECT
        form_ans_bridge_src_id,
        form_ans_bridge_src_sys_inst_src_id,
        form_ans_bridge_form_ques_src_name,
        MAX(form_ans_bridge_form_ans_conformed_answer) AS form_ans_bridge_form_ans_conformed_answer
    FROM silver_rdm_form_answer_bridging
    WHERE form_ans_bridge_src_sys_inst_src_id LIKE 'SONE%'
    GROUP BY
        form_ans_bridge_src_id,
        form_ans_bridge_src_sys_inst_src_id,
        form_ans_bridge_form_ques_src_name
)

SELECT
    sc.ctv3_code,
    sc.ctv3_text,
    sc.id_organisation_source,
    CONCAT('SONE', sc.id_organisation_source) AS sc_source_instance,
    COALESCE(selc.question_heading, dermc.question_heading) AS src_question,

    br.form_ans_bridge_src_id,
    br.form_ans_bridge_src_sys_inst_src_id,
    br.form_ans_bridge_form_ques_src_name,
    br.form_ans_bridge_form_ans_conformed_answer,

    CONCAT_WS(' - ', sc.ctv3_code, br.form_ans_bridge_form_ans_conformed_answer) AS form_ans_src_answer
FROM silver_sone_srcode sc
LEFT JOIN silver_rdm_derm_read_codes dermc
    ON sc.ctv3_code = dermc.code
LEFT JOIN silver_rdm_sel_read_codes selc
    ON sc.ctv3_code = selc.code
LEFT JOIN br_dedup br
    ON TRIM(LOWER(sc.ctv3_code)) = TRIM(LOWER(br.form_ans_bridge_src_id))
   AND TRIM(LOWER(CONCAT('SONE', sc.id_organisation_source))) = TRIM(LOWER(br.form_ans_bridge_src_sys_inst_src_id))
   AND TRIM(LOWER(COALESCE(selc.question_heading, dermc.question_heading))) = TRIM(LOWER(br.form_ans_bridge_form_ques_src_name))
WHERE sc.ctv3_code IS NOT NULL
  AND br.form_ans_bridge_form_ans_conformed_answer IS NOT NULL
LIMIT 100;

In [ ]:
%%sql

WITH br_dedup AS (
    SELECT
        form_ans_bridge_src_id,
        form_ans_bridge_src_sys_inst_src_id,
        form_ans_bridge_form_ques_src_name,
        MAX(form_ans_bridge_form_ans_conformed_answer) AS form_ans_bridge_form_ans_conformed_answer
    FROM silver_rdm_form_answer_bridging
    WHERE form_ans_bridge_src_sys_inst_src_id LIKE 'SONE%'
    GROUP BY
        form_ans_bridge_src_id,
        form_ans_bridge_src_sys_inst_src_id,
        form_ans_bridge_form_ques_src_name
)

SELECT
    COUNT(*) AS joined_rows,
    COUNT(DISTINCT CONCAT(sc.id, '|', sc.ctv3_code, '|', sc.id_organisation_source)) AS distinct_srcode_rows
FROM silver_sone_srcode sc
LEFT JOIN silver_rdm_derm_read_codes dermc
    ON sc.ctv3_code = dermc.code
LEFT JOIN silver_rdm_sel_read_codes selc
    ON sc.ctv3_code = selc.code
INNER JOIN br_dedup br
    ON TRIM(LOWER(sc.ctv3_code)) = TRIM(LOWER(br.form_ans_bridge_src_id))
   AND TRIM(LOWER(CONCAT('SONE', sc.id_organisation_source))) = TRIM(LOWER(br.form_ans_bridge_src_sys_inst_src_id))
   AND TRIM(LOWER(COALESCE(selc.question_heading, dermc.question_heading))) = TRIM(LOWER(br.form_ans_bridge_form_ques_src_name))
WHERE sc.ctv3_code IS NOT NULL;

In [ ]:
%%sql

SELECT
    code,
    COUNT(*) AS row_count,
    COUNT(DISTINCT question_heading) AS distinct_question_heading_count
FROM silver_rdm_derm_read_codes
GROUP BY code
HAVING COUNT(*) > 1
ORDER BY row_count DESC;

In [ ]:
%%sql

SELECT
    code,
    COUNT(*) AS row_count,
    COUNT(DISTINCT question_heading) AS distinct_question_heading_count
FROM silver_rdm_sel_read_codes
GROUP BY code
HAVING COUNT(*) > 1
ORDER BY row_count DESC;

In [ ]:
%%sql

SELECT
    code,
    COUNT(*) AS row_count,
    COUNT(DISTINCT question_heading) AS distinct_question_heading_count
FROM silver_rdm_sel_read_codes
GROUP BY code
HAVING COUNT(*) > 1
ORDER BY row_count DESC;

In [ ]:
%%sql

WITH br_dedup AS (
    SELECT
        form_ans_bridge_src_id,
        form_ans_bridge_src_sys_inst_src_id,
        form_ans_bridge_form_ques_src_name,
        MAX(form_ans_bridge_form_ans_conformed_answer) AS form_ans_bridge_form_ans_conformed_answer
    FROM silver_rdm_form_answer_bridging
    WHERE form_ans_bridge_src_sys_inst_src_id LIKE 'SONE%'
    GROUP BY
        form_ans_bridge_src_id,
        form_ans_bridge_src_sys_inst_src_id,
        form_ans_bridge_form_ques_src_name
)

SELECT
    sc.id,
    sc.ctv3_code,
    sc.id_organisation_source,
    COUNT(*) AS row_count
FROM silver_sone_srcode sc
LEFT JOIN silver_rdm_derm_read_codes dermc
    ON sc.ctv3_code = dermc.code
LEFT JOIN silver_rdm_sel_read_codes selc
    ON sc.ctv3_code = selc.code
INNER JOIN br_dedup br
    ON TRIM(LOWER(sc.ctv3_code)) = TRIM(LOWER(br.form_ans_bridge_src_id))
   AND TRIM(LOWER(CONCAT('SONE', sc.id_organisation_source))) = TRIM(LOWER(br.form_ans_bridge_src_sys_inst_src_id))
   AND TRIM(LOWER(COALESCE(selc.question_heading, dermc.question_heading))) = TRIM(LOWER(br.form_ans_bridge_form_ques_src_name))
WHERE sc.ctv3_code IS NOT NULL
GROUP BY
    sc.id,
    sc.ctv3_code,
    sc.id_organisation_source
HAVING COUNT(*) > 1
ORDER BY row_count DESC
LIMIT 100;

In [ ]:
%%sql

SELECT
    sc.ctv3_code AS srcode_ctv3_code,
    br.form_ans_bridge_src_id AS bridge_src_id,

    CASE
        WHEN TRIM(LOWER(sc.ctv3_code)) = TRIM(LOWER(br.form_ans_bridge_src_id))
        THEN 'MATCH'
        ELSE 'NOT MATCH'
    END AS code_match_check,

    CONCAT('SONE', sc.id_organisation_source) AS srcode_source_instance,
    br.form_ans_bridge_src_sys_inst_src_id AS bridge_source_instance,

    CASE
        WHEN TRIM(LOWER(CONCAT('SONE', sc.id_organisation_source))) = TRIM(LOWER(br.form_ans_bridge_src_sys_inst_src_id))
        THEN 'MATCH'
        ELSE 'NOT MATCH'
    END AS instance_match_check,

    COALESCE(selc.question_heading, dermc.question_heading) AS srcode_question_heading,
    br.form_ans_bridge_form_ques_src_name AS bridge_question_name,

    CASE
        WHEN TRIM(LOWER(COALESCE(selc.question_heading, dermc.question_heading))) = TRIM(LOWER(br.form_ans_bridge_form_ques_src_name))
        THEN 'MATCH'
        ELSE 'NOT MATCH'
    END AS question_match_check,

    br.form_ans_bridge_form_ans_conformed_answer,
    CONCAT_WS(' - ', sc.ctv3_code, br.form_ans_bridge_form_ans_conformed_answer) AS expected_form_ans_src_answer

FROM silver_sone_srcode sc

LEFT JOIN silver_rdm_derm_read_codes dermc
    ON sc.ctv3_code = dermc.code

LEFT JOIN silver_rdm_sel_read_codes selc
    ON sc.ctv3_code = selc.code

LEFT JOIN silver_rdm_form_answer_bridging br
    ON TRIM(LOWER(sc.ctv3_code)) = TRIM(LOWER(br.form_ans_bridge_src_id))
   AND TRIM(LOWER(CONCAT('SONE', sc.id_organisation_source))) = TRIM(LOWER(br.form_ans_bridge_src_sys_inst_src_id))
   AND TRIM(LOWER(COALESCE(selc.question_heading, dermc.question_heading))) = TRIM(LOWER(br.form_ans_bridge_form_ques_src_name))

WHERE sc.ctv3_code IS NOT NULL
  AND br.form_ans_bridge_src_id IS NOT NULL

LIMIT 100;